## 1. Instalar yfinance

In [1]:
#!pip install yfinance

## 2. Imports y configuracion

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

TICKERS = ["NVDA", "GOOGL", "MSFT"]
PERIODO_INICIO = "2011-01-01"
PERIODO_FIN = "2025-12-31"
CARPETA_SALIDA = "datos"

os.makedirs(CARPETA_SALIDA, exist_ok=True)
print(f"Acciones: {TICKERS}")
print(f"Periodo: {PERIODO_INICIO} a {PERIODO_FIN}")

Acciones: ['NVDA', 'GOOGL', 'MSFT']
Periodo: 2011-01-01 a 2025-12-31


## 3. Funciones para indicadores tecnicos

In [3]:
def calcular_sma(close, periodo):
    """Simple Moving Average (Paper 2)"""
    return close.rolling(window=periodo).mean()


def calcular_wma(close, periodo):
    """Weighted Moving Average (Paper 2)"""
    pesos = np.arange(1, periodo + 1)
    return close.rolling(window=periodo).apply(
        lambda x: np.dot(x, pesos) / pesos.sum(), raw=True
    )


def calcular_rsi(close, periodo=14):
    """Relative Strength Index (Paper 2 y Paper 3)"""
    delta = close.diff()
    ganancia = delta.where(delta > 0, 0.0)
    perdida = -delta.where(delta < 0, 0.0)
    avg_ganancia = ganancia.rolling(window=periodo).mean()
    avg_perdida = perdida.rolling(window=periodo).mean()
    rs = avg_ganancia / avg_perdida
    return 100 - (100 / (1 + rs))


def calcular_macd(close):
    """MACD: EMA(12) - EMA(26) (Paper 2)"""
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    return macd


def calcular_macd_signal(macd, periodo=9):
    """Linea de senal del MACD: EMA(9) del MACD (Appel, 2005)"""
    return macd.ewm(span=periodo, adjust=False).mean()


def calcular_bollinger(close, periodo=20, num_std=2):
    """Bandas de Bollinger sobre SMA(20) +/- 2*sigma(20) (Murphy, 1999)"""
    media = close.rolling(window=periodo).mean()
    std = close.rolling(window=periodo).std()
    banda_superior = media + num_std * std
    banda_inferior = media - num_std * std
    return banda_superior, media, banda_inferior


def calcular_momentum(close, periodo=10):
    """Momentum: precio actual - precio hace N dias (Paper 2)"""
    return close - close.shift(periodo)


def calcular_stochastic(high, low, close, k_periodo=14, d_periodo=3):
    """Stochastic K y D (Paper 2)"""
    min_low = low.rolling(window=k_periodo).min()
    max_high = high.rolling(window=k_periodo).max()
    k = 100 * (close - min_low) / (max_high - min_low)
    d = k.rolling(window=d_periodo).mean()
    return k, d


print("Funciones cargadas.")

Funciones cargadas.


## 4. Descargar y procesar cada accion

In [4]:
def descargar_contexto_mercado(inicio, fin):
    """
    Descarga 3 features externas de contexto de mercado (Hu & Luo, 2024):
      - SP500_ret  : retorno diario S&P 500 (^GSPC)   — benchmark general
      - NASDAQ_ret : retorno diario NASDAQ 100 (^NDX)  — índice tecnológico
      - Gold_ret   : retorno diario Oro (GLD)          — activo refugio
    """
    print("\n[CONTEXTO MERCADO] Descargando S&P 500, NASDAQ y Oro...")

    activos = [("^GSPC", "SP500_ret"), ("^NDX", "NASDAQ_ret"), ("GLD", "Gold_ret")]
    df = pd.DataFrame()

    for ticker, col in activos:
        datos = yf.Ticker(ticker).history(start=inicio, end=fin, auto_adjust=True)
        if datos.index.tz is not None:
            datos.index = datos.index.tz_convert(None)
        df[col] = datos["Close"].pct_change()
        print(f"  {col}: {datos['Close'].notna().sum()} filas descargadas")

    print(f"  Columnas: {list(df.columns)}")
    return df


def procesar_accion(ticker, df_contexto):
    print(f"\n{'='*60}")
    print(f"Procesando {ticker}...")
    print(f"{'='*60}")

    # --- 1. Descargar OHLCV ---
    print(f"[1/4] Descargando datos OHLCV...")
    accion = yf.Ticker(ticker)
    df = accion.history(start=PERIODO_INICIO, end=PERIODO_FIN, auto_adjust=True)

    if df.empty:
        print(f"ERROR: No se encontraron datos para {ticker}")
        return None

    df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
    if df.index.tz is not None:
        df.index = df.index.tz_convert(None)
    else:
        df.index = df.index.tz_localize(None)
    df.index.name = "Date"

    print(f"  Filas descargadas: {len(df)}")
    print(f"  Rango: {df.index[0].strftime('%Y-%m-%d')} a {df.index[-1].strftime('%Y-%m-%d')}")

    # --- 2. Calcular indicadores tecnicos ---
    print(f"[2/4] Calculando indicadores tecnicos...")

    df["SMA_20"] = calcular_sma(df["Close"], 20)
    df["SMA_50"] = calcular_sma(df["Close"], 50)
    df["WMA_20"] = calcular_wma(df["Close"], 20)
    df["RSI_14"] = calcular_rsi(df["Close"], 14)
    df["MACD"] = calcular_macd(df["Close"])
    df["MACD_signal"] = calcular_macd_signal(df["MACD"], 9)
    df["BB_upper"], df["BB_mid"], df["BB_lower"] = calcular_bollinger(df["Close"], 20, 2)
    df["Momentum_10"] = calcular_momentum(df["Close"], 10)
    df["Stoch_K"], df["Stoch_D"] = calcular_stochastic(df["High"], df["Low"], df["Close"])

    print(f"  SMA_20, SMA_50, WMA_20, RSI_14, MACD, MACD_signal, BB_upper, BB_mid, BB_lower, Momentum_10, Stoch_K, Stoch_D")

    # --- 3. Agregar features externas ---
    print(f"[3/4] Agregando S&P 500, NASDAQ y Oro...")
    df = df.join(df_contexto, how="left")
    features_ext = ["SP500_ret", "NASDAQ_ret", "Gold_ret"]
    nans_ext = df[features_ext].isnull().sum()
    if nans_ext.any():
        print(f"  NaN en primeras filas (esperado): {nans_ext[nans_ext > 0].to_dict()}")

    # --- 4. Variable objetivo ---
    print(f"[4/4] Calculando variable objetivo...")
    df["Target_Retorno_1d"] = df["Close"].pct_change(1).shift(-1)
    df["Target_Direccion"] = (df["Target_Retorno_1d"] > 0).astype(int)

    # Eliminar filas con NaN
    filas_antes = len(df)
    df = df.dropna()
    print(f"  Filas eliminadas por NaN: {filas_antes - len(df)}")
    print(f"  Filas finales: {len(df)}")

    return df

In [5]:
df_contexto = descargar_contexto_mercado(PERIODO_INICIO, PERIODO_FIN)

resultados = {}
for ticker in TICKERS:
    df = procesar_accion(ticker, df_contexto)
    if df is not None:
        resultados[ticker] = df


[CONTEXTO MERCADO] Descargando S&P 500, NASDAQ y Oro...
  SP500_ret: 3771 filas descargadas
  NASDAQ_ret: 3771 filas descargadas
  Gold_ret: 3771 filas descargadas
  Columnas: ['SP500_ret', 'NASDAQ_ret', 'Gold_ret']

Procesando NVDA...
[1/4] Descargando datos OHLCV...
  Filas descargadas: 3771
  Rango: 2011-01-03 a 2025-12-30
[2/4] Calculando indicadores tecnicos...
  SMA_20, SMA_50, WMA_20, RSI_14, MACD, MACD_signal, BB_upper, BB_mid, BB_lower, Momentum_10, Stoch_K, Stoch_D
[3/4] Agregando S&P 500, NASDAQ y Oro...
  NaN en primeras filas (esperado): {'SP500_ret': 1, 'NASDAQ_ret': 1, 'Gold_ret': 1}
[4/4] Calculando variable objetivo...
  Filas eliminadas por NaN: 50
  Filas finales: 3721

Procesando GOOGL...
[1/4] Descargando datos OHLCV...
  Filas descargadas: 3771
  Rango: 2011-01-03 a 2025-12-30
[2/4] Calculando indicadores tecnicos...
  SMA_20, SMA_50, WMA_20, RSI_14, MACD, MACD_signal, BB_upper, BB_mid, BB_lower, Momentum_10, Stoch_K, Stoch_D
[3/4] Agregando S&P 500, NASDAQ y Oro

## 5. Guardar CSVs

In [6]:
carpeta_abs = os.path.join(os.path.dirname(os.path.abspath("__file__")), "datos")
os.makedirs(carpeta_abs, exist_ok=True)

for ticker, df in resultados.items():
    archivo = os.path.join(carpeta_abs, f"{ticker}_dataset.csv")
    df.to_csv(archivo)
    tamano = os.path.getsize(archivo) / 1024
    print(f"{ticker}_dataset.csv  ->  {tamano:.1f} KB  ({len(df)} filas x {len(df.columns)} columnas)")

print(f"\nCarpeta: {carpeta_abs}")

NVDA_dataset.csv  ->  1531.7 KB  (3721 filas x 22 columnas)
GOOGL_dataset.csv  ->  1509.6 KB  (3721 filas x 22 columnas)
MSFT_dataset.csv  ->  1503.6 KB  (3721 filas x 22 columnas)

Carpeta: c:\Users\hamga\Documents\repo\ai-stock-price-prediction\datos


## 6. Resumen y verificacion

In [7]:
print(f"{'='*60}")
print("RESUMEN FINAL")
print(f"{'='*60}")

for ticker, df in resultados.items():
    print(f"\n--- {ticker} ---")
    print(f"  Filas: {len(df)} (sin NaN)")
    print(f"  Columnas ({len(df.columns)}):")
    for col in df.columns:
        print(f"    - {col}: {df[col].notna().sum()} valores")

RESUMEN FINAL

--- NVDA ---
  Filas: 3721 (sin NaN)
  Columnas (22):
    - Open: 3721 valores
    - High: 3721 valores
    - Low: 3721 valores
    - Close: 3721 valores
    - Volume: 3721 valores
    - SMA_20: 3721 valores
    - SMA_50: 3721 valores
    - WMA_20: 3721 valores
    - RSI_14: 3721 valores
    - MACD: 3721 valores
    - MACD_signal: 3721 valores
    - BB_upper: 3721 valores
    - BB_mid: 3721 valores
    - BB_lower: 3721 valores
    - Momentum_10: 3721 valores
    - Stoch_K: 3721 valores
    - Stoch_D: 3721 valores
    - SP500_ret: 3721 valores
    - NASDAQ_ret: 3721 valores
    - Gold_ret: 3721 valores
    - Target_Retorno_1d: 3721 valores
    - Target_Direccion: 3721 valores

--- GOOGL ---
  Filas: 3721 (sin NaN)
  Columnas (22):
    - Open: 3721 valores
    - High: 3721 valores
    - Low: 3721 valores
    - Close: 3721 valores
    - Volume: 3721 valores
    - SMA_20: 3721 valores
    - SMA_50: 3721 valores
    - WMA_20: 3721 valores
    - RSI_14: 3721 valores
    - MACD

## 7. Vista previa

In [8]:
for ticker, df in resultados.items():
    print(f"\n{'='*40}")
    print(f"{ticker} - Primeras 5 filas")
    print(f"{'='*40}")
    display(df.head())


NVDA - Primeras 5 filas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_mid,BB_lower,Momentum_10,Stoch_K,Stoch_D,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
Date,,,,,,,,,,,,,,,,,,,,,
2011-03-15 04:00:00,0.396739,0.411636,0.389863,0.404761,1256280000,0.488199,0.505387,0.461089,19.394817,-0.028884,...,0.488199,0.378268,-0.091449,10.000024,8.215830,-0.011200,-0.013576,-0.018652,-0.007361,0
2011-03-16 04:00:00,0.401093,0.420575,0.396509,0.401781,1476520000,0.482447,0.506171,0.452859,12.682226,-0.030533,...,0.482447,0.366925,-0.073801,7.999984,9.076936,-0.019495,-0.025071,-0.000220,0.018825,1
2011-03-17 04:00:00,0.412553,0.413241,0.394217,0.409344,1238516000,0.476121,0.507129,0.445897,10.863068,-0.030874,...,0.476121,0.359069,-0.068988,13.076871,10.358960,0.013398,0.010109,0.005358,-0.013438,0
2011-03-18 04:00:00,0.415762,0.417137,0.403385,0.403844,886960000,0.466884,0.507422,0.439013,11.230761,-0.031227,...,0.466884,0.358352,-0.071968,10.200606,10.425820,0.004310,-0.001874,0.010221,0.007946,1
2011-03-21 04:00:00,0.412553,0.416679,0.402927,0.407052,751832000,0.457865,0.506702,0.433315,15.452944,-0.030893,...,0.457865,0.362308,-0.062112,14.940199,12.739226,0.014986,0.018743,0.005565,-0.017455,0



GOOGL - Primeras 5 filas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_mid,BB_lower,Momentum_10,Stoch_K,Stoch_D,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
Date,,,,,,,,,,,,,,,,,,,,,
2011-03-15 04:00:00,13.837086,14.172155,13.787445,14.136414,160063776,14.943472,15.146590,14.713760,20.094425,-0.266250,...,14.943472,14.042143,-0.774383,22.065272,10.821091,-0.011200,-0.013576,-0.018652,-0.021877,0
2011-03-16 04:00:00,14.097944,14.142124,13.682708,13.827158,151788060,14.860263,15.123135,14.607445,17.585771,-0.308382,...,14.860263,13.869102,-1.084384,8.566290,11.493069,-0.019495,-0.025071,-0.000220,0.007647,1
2011-03-17 04:00:00,14.010331,14.122516,13.912541,13.932893,115856028,14.782254,15.102902,14.519124,20.610967,-0.329442,...,14.782254,13.755839,-1.196319,14.836649,15.156070,0.013398,0.010109,0.005358,-0.000534,0
2011-03-18 04:00:00,14.014301,14.097449,13.892685,13.925447,131812056,14.702582,15.079070,14.437523,17.189100,-0.342782,...,14.702582,13.669569,-0.981875,14.395064,12.599334,0.004310,-0.001874,0.010221,0.027519,1
2011-03-21 04:00:00,14.152798,14.390573,14.123014,14.308666,120715164,14.636090,15.060703,14.400007,35.307686,-0.318757,...,14.636090,13.689119,-0.376269,41.886761,23.706158,0.014986,0.018743,0.005565,0.001422,1



MSFT - Primeras 5 filas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_mid,BB_lower,Momentum_10,Stoch_K,Stoch_D,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
Date,,,,,,,,,,,,,,,,,,,,,
2011-03-15 04:00:00,19.202821,19.501429,19.141568,19.440176,76067300,20.114344,20.895706,19.891518,28.571581,-0.394744,...,20.114344,19.246238,-0.589563,18.932055,19.224548,-0.011200,-0.013576,-0.018652,-0.023631,0
2011-03-16 04:00:00,19.310013,19.355954,18.896556,18.980780,100725400,20.031270,20.849374,19.783560,19.255153,-0.433706,...,20.031270,19.063595,-0.987705,4.845853,14.553780,-0.019495,-0.025071,-0.000220,-0.000403,0
2011-03-17 04:00:00,19.187510,19.310016,18.950155,18.973125,62497000,19.945516,20.801215,19.682784,20.598554,-0.459901,...,19.945516,18.920677,-1.087236,4.587305,9.455071,0.013398,0.010109,0.005358,0.000807,1
2011-03-18 04:00:00,19.187512,19.279392,18.988440,18.988440,85486700,19.853254,20.754732,19.591634,20.333441,-0.473961,...,19.853254,18.832850,-0.880508,5.714509,5.049222,0.004310,-0.001874,0.010221,0.021371,1
2011-03-21 04:00:00,19.279390,19.585655,19.256420,19.394239,46878100,19.787024,20.703882,19.547918,36.656000,-0.447203,...,19.787024,18.833411,-0.298611,37.790954,16.030922,0.014986,0.018743,0.005565,-0.001185,0


In [9]:
for ticker, df in resultados.items():
    print(f"\n{'='*40}")
    print(f"{ticker} - Estadisticas descriptivas")
    print(f"{'='*40}")
    display(df.describe())


NVDA - Estadisticas descriptivas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_mid,BB_lower,Momentum_10,Stoch_K,Stoch_D,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
count,3721.000000,3721.000000,3721.000000,3721.000000,3.721000e+03,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,...,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000
mean,24.385390,24.795082,23.926178,24.383378,4.496793e+08,23.917939,23.175550,24.071357,55.239183,0.343081,...,23.917939,21.563399,0.487965,58.440297,58.420004,0.000509,0.000736,0.000333,0.002056,0.529965
std,45.140213,45.826284,44.321390,45.110543,2.619554e+08,44.340014,43.062459,44.581360,16.856555,1.504818,...,44.340014,40.488095,4.194616,30.298307,28.205225,0.010924,0.013225,0.009949,0.028573,0.499168
min,0.264034,0.266555,0.255554,0.260825,4.564400e+07,0.273492,0.280945,0.274471,2.978652,-6.256588,...,0.273492,0.255552,-29.530334,0.000000,0.946960,-0.119841,-0.121932,-0.087808,-0.187559,0.000000
25%,0.472349,0.479277,0.466764,0.471708,2.769320e+08,0.475719,0.471339,0.476509,42.856958,-0.005301,...,0.475719,0.445586,-0.037990,32.098667,33.131252,-0.003779,-0.004880,-0.004804,-0.011949,0.000000
50%,4.573795,4.679528,4.504637,4.584127,3.936940e+08,4.509900,4.380964,4.514283,55.197228,0.011755,...,4.509900,4.149730,0.013970,63.407440,62.948864,0.000688,0.001157,0.000505,0.001833,1.000000
75%,19.597110,19.908752,19.129815,19.556391,5.486480e+08,19.513376,18.684765,19.501944,67.561467,0.191536,...,19.513376,16.818167,0.380196,85.960666,84.214101,0.005675,0.007415,0.005549,0.016283,1.000000
max,208.057150,212.166717,205.537422,207.017273,3.692928e+09,193.373268,187.287456,194.875129,99.428887,9.771465,...,193.373268,177.542753,29.575722,100.000000,99.205181,0.095154,0.120223,0.049038,0.298067,1.000000



GOOGL - Estadisticas descriptivas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_mid,BB_lower,Momentum_10,Stoch_K,Stoch_D,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
count,3721.000000,3721.000000,3721.000000,3721.000000,3.721000e+03,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,...,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000
mean,74.642555,75.446639,73.855722,74.672925,4.940828e+07,73.917727,72.764406,74.169511,54.738144,0.541228,...,73.917727,69.553713,0.789426,57.954674,57.937362,0.000509,0.000736,0.000333,0.000986,0.532384
std,58.833994,59.536376,58.142199,58.858734,3.997943e+07,57.625616,55.742213,58.028785,16.385213,2.086111,...,57.625616,53.676161,5.387084,30.059526,27.859543,0.010924,0.013225,0.009949,0.017490,0.499017
min,11.764627,11.932162,11.740303,11.786469,9.312000e+06,12.419947,12.858574,12.259562,10.962735,-7.034281,...,12.419947,11.133080,-25.770645,0.000000,0.537579,-0.119841,-0.121932,-0.087808,-0.116341,0.000000
25%,28.060083,28.177612,27.728037,27.985699,2.675400e+07,27.844218,27.767117,27.896386,42.407864,-0.179856,...,27.844218,26.727813,-0.827164,32.447853,33.640383,-0.003779,-0.004880,-0.004804,-0.007430,0.000000
50%,54.549151,55.213660,53.876211,54.552628,3.512600e+07,54.527978,54.445598,54.410863,55.026563,0.269207,...,54.527978,51.190046,0.349216,62.410224,61.984581,0.000688,0.001157,0.000505,0.001024,1.000000
75%,114.761500,116.347888,113.370987,114.883980,5.814600e+07,113.908895,112.667496,113.833307,66.522318,1.006931,...,113.908895,105.693773,2.086254,85.577600,83.876935,0.005675,0.007415,0.005549,0.009651,1.000000
max,325.767421,328.383862,318.736980,323.001190,5.611863e+08,313.436005,293.921053,313.874727,98.093749,14.457856,...,313.436005,300.025599,41.553528,100.000000,98.982431,0.095154,0.120223,0.049038,0.162584,1.000000



MSFT - Estadisticas descriptivas


,Open,High,Low,Close,Volume,SMA_20,SMA_50,WMA_20,RSI_14,MACD,...,BB_mid,BB_lower,Momentum_10,Stoch_K,Stoch_D,SP500_ret,NASDAQ_ret,Gold_ret,Target_Retorno_1d,Target_Direccion
count,3721.000000,3721.000000,3721.000000,3721.000000,3.721000e+03,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,...,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000,3721.000000
mean,159.497565,160.971738,157.952854,159.535076,3.307108e+07,158.353345,156.460444,158.745989,55.483614,0.878700,...,158.353345,150.564303,1.242343,59.999288,59.983819,0.000509,0.000736,0.000333,0.000998,0.526471
std,145.834443,147.067586,144.488783,145.830327,1.855777e+07,144.972504,143.525078,145.240944,15.688856,3.318949,...,144.972504,138.261916,9.474883,29.395036,27.069563,0.010924,0.013225,0.009949,0.016250,0.499366
min,18.303685,18.504057,18.226616,18.272861,5.855900e+06,18.647406,19.128809,18.572614,6.805914,-11.667052,...,18.647406,17.732240,-47.152466,0.000000,1.988616,-0.119841,-0.121932,-0.087808,-0.147391,0.000000
25%,37.513588,37.849457,37.253258,37.565662,2.148710e+07,37.504478,37.613429,37.588947,44.236308,-0.170545,...,37.504478,35.307826,-0.956924,36.510273,37.866510,-0.003779,-0.004880,-0.004804,-0.006886,0.000000
50%,96.189511,97.021261,94.611841,95.965469,2.810730e+07,95.492162,94.882096,95.520664,55.334935,0.378832,...,95.492162,90.344899,0.547523,65.785309,65.720363,0.000688,0.001157,0.000505,0.000724,1.000000
75%,266.315532,269.278721,262.083789,266.240662,3.892310e+07,260.762233,259.217661,261.743609,67.184104,1.634333,...,260.762233,241.904417,3.450500,86.096296,84.224446,0.005675,0.007415,0.005549,0.009226,1.000000
max,552.023241,552.242002,538.530652,539.825256,3.193179e+08,519.582895,512.314617,521.732270,99.111864,18.595952,...,519.582895,506.282749,76.464630,100.000000,98.881180,0.095154,0.120223,0.049038,0.142169,1.000000
